# Week 4 — Dataset & Model Loading

**Goal:** Load DenseNet-121 and ViT-B/16, run inference on chest X-ray images, and visually inspect predictions.

Run this notebook on **Google Colab** (Runtime → Change runtime type → T4 GPU).

In [ ]:
# ── Install dependencies (Colab only — skip if running locally) ──────────────
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q torchxrayvision transformers timm grad-cam scikit-image seaborn

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import torch
import torchvision.transforms as T
import torchxrayvision as xrv
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
from transformers import ViTForImageClassification, ViTImageProcessor

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : ", end="")
device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(device)

## 1. Load DenseNet-121 (via torchxrayvision)

`torchxrayvision` ships pretrained DenseNet-121 weights trained on CheXpert.
The model outputs probabilities for 18 pathologies.

In [ ]:
densenet = xrv.models.DenseNet(weights="densenet121-res224-chex")
densenet.eval().to(device)

print("DenseNet-121 pathology labels:")
for i, label in enumerate(densenet.pathologies):
    print(f"  {i:2d}. {label}")

## 2. Load ViT-B/16 (via HuggingFace)

We use the same 5 pathology classes that CheXpert labels most reliably:
`Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion`.

> **Note:** This loads the base ViT-B/16 pretrained on ImageNet-21k with a randomly initialised classification head. The predictions will not be meaningful until the head is trained on CheXpert labels (Week 4 extension or Week 8). The loading + inference pipeline is what matters here.

In [ ]:
PATHOLOGIES = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]

VIT_CHECKPOINT = "google/vit-base-patch16-224-in21k"
vit_processor = ViTImageProcessor.from_pretrained(VIT_CHECKPOINT)
vit_model = ViTForImageClassification.from_pretrained(
    VIT_CHECKPOINT,
    num_labels=len(PATHOLOGIES),
    ignore_mismatched_sizes=True,
    id2label={i: p for i, p in enumerate(PATHOLOGIES)},
    label2id={p: i for i, p in enumerate(PATHOLOGIES)},
)
vit_model.eval().to(device)
print(f"ViT-B/16 loaded  |  output classes: {vit_model.config.num_labels}")

## 3. Load Sample Images

We use `torchxrayvision`'s built-in demo image to test the pipeline before your CheXpert approval arrives.
Once CheXpert is approved, replace `get_sample_images()` with a loader over your downloaded subset.

In [ ]:
def get_sample_images(n: int = 10) -> list[np.ndarray]:
    """Return n copies of the torchxrayvision demo image (grayscale, 224x224)."""
    demo = xrv.datasets.utils.normalize(
        xrv.datasets.utils.resize(xrv.datasets.XRayCenterCrop()(xrv.datasets.XRayResizer(224)(xrv.datasets.utils.get_sample_data()['PA'])),
        224),
        maxval=255, reshape=True
    )
    return [demo.copy() for _ in range(n)]


def load_chexpert_samples(folder: str, n: int = 10) -> list[np.ndarray]:
    """Load n images from a CheXpert folder once your dataset is approved."""
    paths = sorted(Path(folder).rglob("*.jpg"))[:n]
    images = []
    for p in paths:
        img = Image.open(p).convert("L")          # grayscale
        img = img.resize((224, 224))
        arr = np.array(img, dtype=np.float32)
        arr = xrv.datasets.utils.normalize(arr, maxval=255, reshape=True)
        images.append(arr)
    return images


# ── Use demo images until CheXpert is available ───────────────────────────────
# images = load_chexpert_samples("../data/CheXpert-v1.0-small/train", n=10)
images = get_sample_images(n=10)
print(f"Loaded {len(images)} images  |  shape: {images[0].shape}  |  dtype: {images[0].dtype}")

## 4. DenseNet-121 Inference

In [ ]:
def predict_densenet(model, image_arr: np.ndarray) -> dict[str, float]:
    """Run DenseNet inference. Returns {pathology: probability}."""
    tensor = torch.from_numpy(image_arr).unsqueeze(0).to(device)  # (1,1,H,W)
    with torch.no_grad():
        logits = model(tensor)
    probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    return dict(zip(model.pathologies, probs))


print("DenseNet-121 predictions (top-5 per image)\n" + "-"*50)
densenet_preds = []
for i, img in enumerate(images):
    pred = predict_densenet(densenet, img)
    densenet_preds.append(pred)
    top5 = sorted(pred.items(), key=lambda x: x[1], reverse=True)[:5]
    top5_str = "  |  ".join(f"{k}: {v:.3f}" for k, v in top5)
    print(f"Image {i+1:2d}: {top5_str}")

## 5. ViT-B/16 Inference

In [ ]:
def predict_vit(model, processor, image_arr: np.ndarray) -> dict[str, float]:
    """Run ViT inference. Converts grayscale→RGB for the processor."""
    # ViT processor expects RGB PIL image
    arr_uint8 = ((image_arr[0] + 1) * 127.5).clip(0, 255).astype(np.uint8)
    pil_rgb = Image.fromarray(arr_uint8).convert("RGB")
    inputs = processor(images=pil_rgb, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    return dict(zip(PATHOLOGIES, probs))


print("ViT-B/16 predictions (untrained head — values are random)\n" + "-"*50)
vit_preds = []
for i, img in enumerate(images):
    pred = predict_vit(vit_model, vit_processor, img)
    vit_preds.append(pred)
    row = "  |  ".join(f"{k}: {v:.3f}" for k, v in pred.items())
    print(f"Image {i+1:2d}: {row}")

## 6. Comparison Plot — DenseNet vs ViT on the 5 shared pathologies

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
fig.suptitle("DenseNet-121 vs ViT-B/16 — pathology probabilities (10 sample images)", fontsize=13)

for i in range(10):
    ax = axes[i // 5][i % 5]
    dn_vals = [densenet_preds[i].get(p, 0.0) for p in PATHOLOGIES]
    vit_vals = [vit_preds[i][p] for p in PATHOLOGIES]
    x = np.arange(len(PATHOLOGIES))
    w = 0.35
    ax.bar(x - w/2, dn_vals, w, label="DenseNet", color="steelblue", alpha=0.85)
    ax.bar(x + w/2, vit_vals, w, label="ViT",      color="tomato",    alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([p.replace(" ", "\n") for p in PATHOLOGIES], fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_title(f"Image {i+1}", fontsize=9)
    if i == 0:
        ax.legend(fontsize=7)

plt.tight_layout()
Path("../figures").mkdir(exist_ok=True)
plt.savefig("../figures/week4_model_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → figures/week4_model_predictions.png")

## 7. Sanity Check

Answer these before moving to Week 5:

1. **DenseNet predictions** — do any pathology probabilities look unexpectedly high or low on the demo image?
2. **ViT predictions** — values should look random (~0.5 on average) because the head is untrained. That is expected.
3. Both models ran without errors and produced output tensors of the right shape.

Once CheXpert is approved:
- Swap in `load_chexpert_samples()` and re-run
- DenseNet predictions should show plausible probabilities (0.1–0.8 range)
- ViT will need fine-tuning before its predictions are meaningful